# Opion v2 re-distillation — fix the leaky own-action feature (T4 GPU)

**Why:** the original student was trained with ctx[14] = own-action amt/pot.
That feature perfectly predicts "raise vs not" in the training data, so the
model learned the shortcut and CANNOT play live (live test: 0 raises with
amt=0 forced). Fix: re-distill with **ctx[14] zeroed everywhere** — a pure
state-only policy, live-playable with a single forward.

## How to use
1. **Runtime → Change runtime type → T4 GPU**
2. Drag-drop `final.pt` (or `final_act.pt`) into the file panel (3.3MB)
3. Run cells top-to-bottom (~25-50 min). Keepalive runs in the background.
4. Download **`final_act_v2.pt`** at the end and give it back to the chat.

The test drawer stays LOCKED — v2 is judged on val here + live screens later.

In [ ]:
# 1. Runtime check
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "set Runtime -> Change runtime type -> T4 GPU"

In [ ]:
# 2. Keepalive: free -h every 5 min
import threading, time, subprocess

def _ka():
    while True:
        try:
            out = subprocess.run(["free", "-h"], capture_output=True, text=True).stdout
            print("[keepalive]\n" + out, flush=True)
        except Exception as e:
            print("[keepalive] err", e, flush=True)
        time.sleep(300)

threading.Thread(target=_ka, daemon=True).start()
print("keepalive running")

In [ ]:
# 3. Data (same packed.tar as before)
import glob, os, tarfile
if not glob.glob("packed/*/*/acts.npz"):
    if not os.path.exists("packed.tar"):
        import gdown
        gdown.download(id="1AgcO3IHlhFc4czzr1fU_EG39Bh7809TA", output="packed.tar")
    with tarfile.open("packed.tar") as tf:
        tf.extractall("/content")
shards = sorted(glob.glob("packed/*/*/acts.npz"))
assert len(shards) == 40, f"expected 40 shards, found {len(shards)}"
print(len(shards), "shards found")

In [ ]:
# 4. Upload the v1 model: drag-drop final.pt (or final_act.pt) into the panel
from google.colab import files
import os
if not os.path.exists("final.pt"):
    if os.path.exists("final_act.pt"):
        import shutil; shutil.copy("final_act.pt", "final.pt")
    else:
        files.upload()
assert os.path.exists("final.pt"), "upload final.pt / final_act.pt"
print("v1 weights present")

In [ ]:
# 5. Model (byte-exact training architecture)
import torch
import torch.nn as nn

N_TOK, N_CTX, N_ACT, VOCAB = 41, 17, 6, 85


def _map_tokens(t):
    t = t.long()
    return torch.where(t < 53, t, 52 + t.clamp(min=1, max=32))


class Block(nn.Module):
    def __init__(self, d, h, f):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)
        self.h = h
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, f)
        self.fc2 = nn.Linear(f, d)
        self.act = nn.GELU()

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q = q.view(B, T, self.h, -1).transpose(1, 2)
        k = k.view(B, T, self.h, -1).transpose(1, 2)
        v = v.view(B, T, self.h, -1).transpose(1, 2)
        att = torch.softmax(q @ k.transpose(-2, -1) / (k.shape[-1] ** 0.5), -1)
        x = x + self.proj((att @ v).transpose(1, 2).reshape(B, T, D))
        x = x + self.fc2(self.act(self.fc1(self.ln2(x))))
        return x


class OpionPolicy(nn.Module):
    def __init__(self, d=128, layers=4, heads=4, f=512):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(N_TOK, d)
        self.ctx_proj = nn.Sequential(nn.LayerNorm(N_CTX), nn.Linear(N_CTX, d))
        self.blocks = nn.ModuleList(Block(d, heads, f) for _ in range(layers))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, N_ACT)

    def forward(self, tokens, ctx):
        x = self.emb(_map_tokens(tokens)) + self.pos.weight[None, :N_TOK]
        x = x + self.ctx_proj(ctx.float())[:, None, :]
        for b in self.blocks:
            x = b(x)
        return self.head(self.ln_f(x).mean(dim=1))

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


dev = "cuda"
model = OpionPolicy().to(dev)
model.load_state_dict(torch.load("final.pt", map_location="cpu")["model"])
print("loaded v1 | params:", f"{model.n_params():,}")

In [ ]:
# 6. Data on GPU — ctx[:,14] ZEROED everywhere (train + val)
import numpy as np


def load(parts, cap=0):
    cols = {}
    n = 0
    for f in shards:
        z = np.load(f, allow_pickle=False)
        sel = np.isin(z["part"], parts)
        if not sel.any():
            continue
        for k in ("tokens", "ctx", "legal", "p_soft", "act_idx", "street"):
            cols.setdefault(k, []).append(z[k][sel])
        n += int(sel.sum())
    out = {k: np.concatenate(v) for k, v in cols.items()}
    if cap and len(out["act_idx"]) > cap:
        out = {k: v[:cap] for k, v in out.items()}
    out["ctx"][:, 14] = 0.0          # <<< THE FIX: kill the leaky feature
    return out, n


tr, n_tr = load((0,))
va, n_va = load((1,), cap=200_000)


def to_gpu(d):
    t = {"tokens": torch.from_numpy(d["tokens"].astype(np.int64)),
         "ctx": torch.from_numpy(d["ctx"].astype(np.float32)),
         "legal": torch.from_numpy(d["legal"].astype(np.float32)),
         "p_soft": torch.from_numpy(d["p_soft"].astype(np.float32)),
         "act_idx": torch.from_numpy(d["act_idx"].astype(np.int64)),
         "street": torch.from_numpy(d["street"].astype(np.int64))}
    return {k: v.to(dev) for k, v in t.items()}


tr = to_gpu(tr)
va = to_gpu(va)
free, total = torch.cuda.mem_get_info()
print(f"train={n_tr:,} val={len(va['act_idx']):,} | GPU free {free >> 20}MB")


def evaluate(m, d, batch=16384):
    m.eval()
    kl_s = ag_s = ah_s = n = 0.0
    mix = np.zeros(6)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(d["act_idx"]), batch):
            sl = slice(i, i + batch)
            logits = m(d["tokens"][sl], d["ctx"][sl]).float()
            logits = logits.masked_fill(d["legal"][sl] == 0, -1e9)
            q = torch.softmax(logits, -1)
            p = d["p_soft"][sl]
            kl = (p * (torch.log(p + 1e-8) - torch.log(q + 1e-8))).sum(-1).mean()
            ag = (p.argmax(-1) == q.argmax(-1)).float().mean()
            ah = (q.argmax(-1) == d["act_idx"][sl]).float().mean()
            b = len(p)
            kl_s += kl.item() * b
            ag_s += ag.item() * b
            ah_s += ah.item() * b
            n += b
            mix += np.bincount(q.argmax(-1).cpu().numpy(), minlength=6)
    m.train()
    return kl_s / n, ag_s / n, ah_s / n, mix / n


kl, ag, ah, mix = evaluate(model, va)
print(f"BEFORE (ctx14=0 eval): KL={kl:.4f} argmax_ag={ag:.4f} hard_ag={ah:.4f}")
print("BEFORE live bucket mix:", np.round(mix, 3).tolist())
print("teacher reference    : [0.17, 0.44, 0.10, 0.26, 0.04, 0.00]")

In [ ]:
# 7. Fine-tune: soft-CE, 2 epochs, lr 1e-4 (the shortcut must unlearn)
import time
import torch.nn.functional as F

EPOCHS, BATCH, LR = 2, 4096, 1e-4
steps = int(n_tr // BATCH * EPOCHS)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps,
                                            pct_start=0.05)
scaler = torch.amp.GradScaler("cuda")
print(f"steps={steps}")
t0 = time.time()
for step in range(1, steps + 1):
    idx = torch.randint(0, n_tr, (BATCH,), device=dev)
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(tr["tokens"][idx], tr["ctx"][idx]).float()
        logits = logits.masked_fill(tr["legal"][idx] == 0, -1e9)
        logq = torch.log_softmax(logits, -1)
        loss = -(tr["p_soft"][idx] * logq).sum(-1).mean()
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    sched.step()
    if step % 500 == 0 or step == steps:
        with torch.no_grad():
            q = torch.softmax(logits, -1)
            h = (q.argmax(-1) == tr["act_idx"][idx]).float().mean()
        print(f"  step {step}/{steps} loss={loss.item():.4f} "
              f"hard_ag={h.item():.3f} {time.time()-t0:.0f}s", flush=True)
    if step % 3000 == 0 or step == steps:
        kl, ag, ah, mix = evaluate(model, va)
        print(f"  [val] step {step}: KL={kl:.4f} argmax_ag={ag:.4f} "
              f"hard_ag={ah:.4f}", flush=True)

In [ ]:
# 8. Final val report (ctx14=0) — this is the honest live-playable number
kl, ag, ah, mix = evaluate(model, va)
print(f"V2 FINAL: KL={kl:.4f} argmax_ag={ag:.4f} hard_ag={ah:.4f}")
print("V2 live bucket mix:", np.round(mix, 3).tolist())
print("teacher reference : [0.17, 0.44, 0.10, 0.26, 0.04, 0.00]")
by = []
for s in range(4):
    m = va["street"] == s
    d_s = {k: v[m] for k, v in va.items()}
    k2, a2, h2, _ = evaluate(model, d_s, batch=8192)
    by.append({"street": s, "kl": round(k2, 4), "argmax_ag": round(a2, 4),
               "n": int(m.sum())})
    print(f"  street {s}: KL={k2:.4f} argmax_ag={a2:.4f} n={int(m.sum())}")
RESULT = {"kl": round(kl, 4), "argmax_ag": round(ag, 4),
          "hard_ag": round(ah, 4), "by_street": by,
          "mix": [round(float(x), 4) for x in mix]}
import json
print("RESULT " + json.dumps(RESULT))

In [ ]:
# 9. Save + download
torch.save({"model": model.state_dict(), "final": RESULT,
            "note": "v2: ctx[14] zeroed, live-playable"},
           "final_act_v2.pt")
print("saved final_act_v2.pt (", os.path.getsize("final_act_v2.pt") / 1e6, "MB )")
from google.colab import files
files.download("final_act_v2.pt")
print("V2_DONE")